# Quantify the Severe Class Imbalance
Before modeling, you need to see exactly how rare failure events are in Q4 2025. This visual will justify why you need an advanced machine learning pipeline later.

In [0]:
from pyspark.sql import functions as F

df = spark.table("backblaze_master_delta")
label_counts = df.groupBy("label").agg(F.count("*").alias("count"))
total_count = df.count()
label_dist = label_counts.withColumn("percentage", F.col("count") / total_count * 100)
display(label_dist)

label,count,percentage
1,21052,0.06803761447170271
0,30920656,99.9319623855283


Databricks visualization. Run in Databricks to view.

# Calculate Annualized Failure Rate (AFR) by Manufacturer
Hard drive reliability varies drastically by brand. To calculate this correctly over millions of rows, you will use the industry standard formula:$$\text{AFR} = \left( \frac{\sum \text{Failures}}{\sum \text{Drive-Days}} \right) \times 365 \times 100\%$$

In [0]:
manufacturer_expr = (
    F.when(F.col("model").startswith("ST"), "Seagate")
     .when(F.col("model").startswith("WDC"), "Western Digital")
     .when(F.col("model").startswith("WD"), "Western Digital")
     .when(F.col("model").startswith("TOSHIBA"), "Toshiba")
     .when(F.col("model").startswith("HGST"), "HGST")
     .otherwise("Other")
)

df_with_manufacturer = df.withColumn("manufacturer", manufacturer_expr)

afr_df = (
    df_with_manufacturer.groupBy("manufacturer")
    .agg(
        F.sum("failure").alias("failures"),
        F.count("*").alias("total_count")
    )
    .withColumn("AFR", (F.col("failures") / F.col("total_count")) * 365 * 100)
    .orderBy(F.col("AFR").desc())
)

display(afr_df)

manufacturer,failures,total_count,AFR
HGST,194,2486929,2.847286754064953
Seagate,408,10313997,1.4438631308502416
Toshiba,247,10267727,0.8780424333447899
Western Digital,95,7564976,0.45836232659561643
Other,1,308079,0.11847610515484666


Databricks visualization. Run in Databricks to view.

# Map the Drive Age Lifecycle (The Bathtub Curve)
Drives tend to fail early due to manufacturing defects ("infant mortality") or much later due to mechanical wear. You want to see if your Q4 2025 failures follow this classic reliability curve.

In [0]:
age_at_failure_df = (
    df.filter(F.col("failure") == 1)
      .select((F.col("smart_9_raw") / 8760).alias("age_at_failure_years"))
)

display(age_at_failure_df)

age_at_failure_years
5.791666666666667
4.946575342465754
4.111187214611872
4.794634703196347
4.660844748858447
2.218607305936073
2.2442922374429224
2.1973744292237445
0.7481735159817352
0.12009132420091324


Databricks visualization. Run in Databricks to view.

# Compare Critical SMART Profiles (Healthy vs. Failing)
Do failing drives actually show different warning signs compared to healthy ones? You need to profile the averages of the four most predictive SMART metrics.

In [0]:
smart_metrics = [
    "smart_5_raw",
    "smart_187_raw",
    "smart_197_raw",
    "smart_198_raw"
]

avg_smart_df = (
    df.groupBy("label")
      .agg(*[F.avg(col).alias(f"avg_{col}") for col in smart_metrics])
      .orderBy("label")
)

display(avg_smart_df)

label,avg_smart_5_raw,avg_smart_187_raw,avg_smart_197_raw,avg_smart_198_raw
0,47.38146710082736,1.6827285294335281,2.487955365500654,1.6440542852648405
1,2330.3838590157707,125.6615523465704,388.4408132243967,348.7464848945468


Databricks visualization. Run in Databricks to view.

# Benchmark the "Backblaze Simple Rule" (The Target to Beat)
Backblaze famously noted that if a drive has a raw value greater than 0 on SMART 5, 187, 197, or 198, it should be replaced. You need to calculate the precision and recall of this simple human heuristic so you know exactly what your Machine Learning model needs to outperform.

In [0]:
baseline_pred_df = df.withColumn(
    "baseline_pred",
    F.when(
        (F.col("smart_5_raw") > 0) |
        (F.col("smart_187_raw") > 0) |
        (F.col("smart_197_raw") > 0) |
        (F.col("smart_198_raw") > 0),
        1
    ).otherwise(0)
)

tp = baseline_pred_df.filter((F.col("baseline_pred") == 1) & (F.col("label") == 1)).count()
fp = baseline_pred_df.filter((F.col("baseline_pred") == 1) & (F.col("label") == 0)).count()
fn = baseline_pred_df.filter((F.col("baseline_pred") == 0) & (F.col("label") == 1)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else None
recall = tp / (tp + fn) if (tp + fn) > 0 else None

print(f"Precision: {precision}")
print(f"Recall: {recall}")

Precision: 0.007158684253789755
Recall: 0.657752232566977
